# Detección de transacciones potencialmente fraudulentas

**Curso:** Inteligencia Artificial y Minirobots · Universidad Nacional de Colombia

El ejercicio tiene dos etapas: generar y limpiar datos sintéticos, y comparar una regresión logística implementada con NumPy (sin scikit-learn) con una regla interpretable obtenida mediante Programación Genética (PG). El notebook reúne ambas etapas para que sea autocontenido en Google Colab: no necesita cargar archivos ni rutas locales.

Ejecuta las celdas en orden. La semilla `2026` hace reproducibles los datos y las particiones. Los datos son sintéticos y se usan únicamente con fines didácticos; no representan transacciones bancarias reales.

In [1]:
%pip -q install numpy pandas deap

import operator
import random

import numpy as np
import pandas as pd
from deap import algorithms, base, creator, gp, tools

SEED = 2026
N_TRANSACTIONS = 5000
FEATURES = [
    'monto_pesos', 'hora_dia', 'distancia_residencia_km',
    'ingreso_mensual_pesos',
]
TARGET = 'fraude'

def train_test_split(frame, test_size, random_state, stratify):
    rng = np.random.default_rng(random_state)
    labels = np.asarray(stratify)
    train_indices = []
    test_indices = []
    for label in np.unique(labels):
        indices = rng.permutation(np.flatnonzero(labels == label))
        n_test = max(1, int(round(len(indices) * test_size)))
        test_indices.extend(indices[:n_test])
        train_indices.extend(indices[n_test:])
    train_indices = rng.permutation(train_indices)
    test_indices = rng.permutation(test_indices)
    return frame.iloc[train_indices].copy(), frame.iloc[test_indices].copy()

def fit_logistic_regression(features, labels, max_iter=1000, C=1.0):
    labels = np.asarray(labels, dtype=float)
    design = np.column_stack([np.ones(len(features)), features])
    class_counts = np.bincount(labels.astype(int), minlength=2)
    sample_weights = len(labels) / (2 * class_counts[labels.astype(int)])
    penalty = np.eye(design.shape[1]) / C
    penalty[0, 0] = 0.0
    coefficients = np.zeros(design.shape[1])
    for _ in range(max_iter):
        probabilities = 1 / (1 + np.exp(-np.clip(design @ coefficients, -35, 35)))
        gradient = design.T @ (sample_weights * (probabilities - labels)) + penalty @ coefficients
        curvature = sample_weights * probabilities * (1 - probabilities)
        hessian = (design.T * curvature) @ design + penalty
        step = np.linalg.solve(hessian, gradient)
        coefficients -= step
        if np.linalg.norm(step) < 1e-8 * (1 + np.linalg.norm(coefficients)):
            break
    return coefficients

def logistic_predict_proba(features, coefficients):
    design = np.column_stack([np.ones(len(features)), features])
    positive_probability = 1 / (1 + np.exp(-np.clip(design @ coefficients, -35, 35)))
    return np.column_stack([1 - positive_probability, positive_probability])

def confusion_matrix(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    return np.array([
        [np.sum((y_true == 0) & (y_pred == 0)), np.sum((y_true == 0) & (y_pred == 1))],
        [np.sum((y_true == 1) & (y_pred == 0)), np.sum((y_true == 1) & (y_pred == 1))],
    ])

def f1_score(y_true, y_pred, zero_division=0):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    denominator = 2 * tp + fp + fn
    return 2 * tp / denominator if denominator else zero_division

def roc_auc_score(y_true, scores):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    n_positive = np.sum(y_true == 1)
    n_negative = np.sum(y_true == 0)
    if not n_positive or not n_negative:
        raise ValueError('ROC-AUC requiere ejemplos de ambas clases.')
    order = np.argsort(scores, kind='mergesort')
    sorted_scores = scores[order]
    ranks = np.empty(len(scores), dtype=float)
    start = 0
    while start < len(scores):
        end = start + 1
        while end < len(scores) and sorted_scores[end] == sorted_scores[start]:
            end += 1
        ranks[order[start:end]] = (start + 1 + end) / 2
        start = end
    positive_rank_sum = ranks[y_true == 1].sum()
    return (positive_rank_sum - n_positive * (n_positive + 1) / 2) / (n_positive * n_negative)

def average_precision_score(y_true, scores):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    n_positive = np.sum(y_true == 1)
    if not n_positive:
        raise ValueError('Average precision requiere ejemplos positivos.')
    order = np.argsort(-scores, kind='mergesort')
    sorted_scores = scores[order]
    sorted_labels = y_true[order]
    true_positives = 0
    false_positives = 0
    previous_recall = 0.0
    average_precision = 0.0
    start = 0
    while start < len(scores):
        end = start + 1
        while end < len(scores) and sorted_scores[end] == sorted_scores[start]:
            end += 1
        true_positives += np.sum(sorted_labels[start:end] == 1)
        false_positives += np.sum(sorted_labels[start:end] == 0)
        recall = true_positives / n_positive
        precision = true_positives / (true_positives + false_positives)
        average_precision += (recall - previous_recall) * precision
        previous_recall = recall
        start = end
    return average_precision

def classification_report(y_true, y_pred, target_names=None, zero_division=0):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    names = target_names or ['0', '1']
    rows = []
    for label, name in enumerate(names):
        true_positive = np.sum((y_true == label) & (y_pred == label))
        false_positive = np.sum((y_true != label) & (y_pred == label))
        false_negative = np.sum((y_true == label) & (y_pred != label))
        support = np.sum(y_true == label)
        precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else zero_division
        recall = true_positive / support if support else zero_division
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else zero_division
        rows.append((name, precision, recall, f1, support))
    accuracy = np.mean(y_true == y_pred)
    supports = np.array([row[4] for row in rows], dtype=float)
    weighted = [np.average([row[index] for row in rows], weights=supports) for index in (1, 2, 3)]
    macro = [np.mean([row[index] for row in rows]) for index in (1, 2, 3)]
    lines = ['              precision    recall  f1-score   support', '']
    lines.extend(f'{name:>12} {precision:>10.2f} {recall:>9.2f} {f1:>9.2f} {support:>9d}'
                 for name, precision, recall, f1, support in rows)
    lines.extend([
        '', f'    accuracy {accuracy:>27.2f} {len(y_true):>9d}',
        f'   macro avg {macro[0]:>10.2f} {macro[1]:>9.2f} {macro[2]:>9.2f} {len(y_true):>9d}',
        f'weighted avg {weighted[0]:>10.2f} {weighted[1]:>9.2f} {weighted[2]:>9.2f} {len(y_true):>9d}',
    ])
    return '\n'.join(lines)

print('Entorno listo')

Note: you may need to restart the kernel to use updated packages.
Entorno listo


## 1. Generación y limpieza de los datos

Se simulan montos, horarios, distancias e ingresos con distribuciones sesgadas. La probabilidad de fraude aumenta con ciertos indicadores de riesgo. A los datos iniciales se les incorporan algunos duplicados y distancias faltantes; después se limpian para crear el conjunto utilizado por los modelos. La imputación de distancias se ajusta más adelante usando solo el conjunto de entrenamiento.

In [2]:
def generate_raw_data(n_rows=N_TRANSACTIONS):
    rng = np.random.default_rng(SEED)
    income = np.clip(rng.lognormal(np.log(2_800_000), 0.55, n_rows), 900_000, 15_000_000)
    hour_probabilities = np.array([
        0.01, 0.005, 0.004, 0.004, 0.005, 0.01, 0.025, 0.05,
        0.07, 0.07, 0.07, 0.07, 0.08, 0.08, 0.075, 0.07,
        0.065, 0.06, 0.06, 0.05, 0.04, 0.03, 0.02, 0.011,
    ])
    hour_probabilities /= hour_probabilities.sum()
    hour = rng.choice(np.arange(24), size=n_rows, p=hour_probabilities)
    distance = np.clip(rng.gamma(shape=2.0, scale=4.0, size=n_rows), 0.1, 80)
    amount = np.clip(
        income * rng.lognormal(np.log(0.025), 0.9, n_rows), 5_000, 8_000_000
    )
    night = ((hour <= 5) | (hour >= 23)).astype(float)
    high_amount = (amount / income > 0.20).astype(float)
    far_away = (distance > 20).astype(float)
    risk_score = -4.2 + 2.2 * night + 1.7 * high_amount + 1.1 * far_away
    risk_score += 0.8 * (amount / income > 0.50)
    fraud_probability = 0.01 + 0.45 / (1 + np.exp(-risk_score))
    fraud = rng.binomial(1, fraud_probability)

    data = pd.DataFrame({
        'monto_pesos': np.rint(amount),
        'hora_dia': hour,
        'distancia_residencia_km': np.round(distance, 2),
        'ingreso_mensual_pesos': np.rint(income),
        'fraude': fraud,
    })
    missing_rows = rng.choice(n_rows, size=round(n_rows * 0.01), replace=False)
    data.loc[missing_rows, 'distancia_residencia_km'] = np.nan
    duplicate_rows = data.sample(round(n_rows * 0.01), random_state=SEED)
    return pd.concat([data, duplicate_rows], ignore_index=True)

def clean_data(data):
    clean = data.drop_duplicates().copy()
    clean['hora_dia'] = clean['hora_dia'].round().clip(0, 23)
    clean['monto_pesos'] = clean['monto_pesos'].clip(5_000, 8_000_000)
    clean['distancia_residencia_km'] = clean['distancia_residencia_km'].clip(0.1, 80)
    clean['ingreso_mensual_pesos'] = clean['ingreso_mensual_pesos'].clip(900_000, 15_000_000)
    clean[TARGET] = clean[TARGET].astype(int).clip(0, 1)
    return clean.reset_index(drop=True)

raw_data = generate_raw_data()
data = clean_data(raw_data)
print(f'Filas iniciales: {len(raw_data):,}')
print(f'Filas después de limpieza: {len(data):,}')
print(f'Tasa de fraude: {data[TARGET].mean():.2%}')
data.head()

Filas iniciales: 5,050
Filas después de limpieza: 5,000
Tasa de fraude: 1.78%


,monto_pesos,hora_dia,distancia_residencia_km,ingreso_mensual_pesos,fraude
0,56876.0,16,7.84,1810136.0,0
1,123969.0,17,3.45,3196107.0,0
2,27448.0,19,8.67,986729.0,0
3,161395.0,12,3.75,6033298.0,0
4,56760.0,6,16.46,3977612.0,0


## 2. Modelo base: regresión logística

La regresión logística proporciona una referencia para comparar con la regla simbólica. Se estandarizan las cuatro variables numéricas y se conserva una partición estratificada de entrenamiento/prueba. La mediana usada para completar distancias faltantes se calcula en entrenamiento y se aplica a ambos conjuntos para evitar filtrar información de prueba.

In [3]:
train, test = train_test_split(
    data, test_size=0.25, random_state=SEED, stratify=data[TARGET]
)
distance_median = train['distancia_residencia_km'].median()
train = train.copy()
test = test.copy()
train['distancia_residencia_km'] = train['distancia_residencia_km'].fillna(distance_median)
test['distancia_residencia_km'] = test['distancia_residencia_km'].fillna(distance_median)
feature_means = train[FEATURES].mean().to_numpy()
feature_scales = train[FEATURES].std(ddof=0).replace(0, 1).to_numpy()
train_features = (train[FEATURES].to_numpy() - feature_means) / feature_scales
test_features = (test[FEATURES].to_numpy() - feature_means) / feature_scales
coefficients = fit_logistic_regression(
    train_features, train[TARGET].to_numpy(), max_iter=1000, C=1.0
)
baseline_probabilities = logistic_predict_proba(test_features, coefficients)[:, 1]
baseline_predictions = (baseline_probabilities >= 0.5).astype(int)
print(f'ROC-AUC: {roc_auc_score(test[TARGET], baseline_probabilities):.3f}')
print(f'Average precision: {average_precision_score(test[TARGET], baseline_probabilities):.3f}')
print('Matriz de confusión [real x predicho]:')
print(confusion_matrix(test[TARGET], baseline_predictions))
print(classification_report(
    test[TARGET], baseline_predictions,
    target_names=['No fraude', 'Fraude'], zero_division=0,
))

ROC-AUC: 0.476
Average precision: 0.018
Matriz de confusión [real x predicho]:
[[621 607]
 [ 13   9]]
              precision    recall  f1-score   support

   No fraude       0.98      0.51      0.67      1228
      Fraude       0.01      0.41      0.03        22

    accuracy                        0.50      1250
   macro avg       0.50      0.46      0.35      1250
weighted avg       0.96      0.50      0.66      1250


## 3. Regla de riesgo con Programación Genética

Las variables continuas se convierten en cinco indicadores booleanos: monto alto (≥20 % del ingreso), horario inusual, distancia lejana, ingreso bajo y monto muy alto (≥50 % del ingreso). La PG busca una expresión con `AND`, `OR`, `XOR` y `NOT`.

La aptitud penaliza los falsos negativos con el doble de peso que los falsos positivos y luego favorece expresiones cortas. La búsqueda usa solo el conjunto de entrenamiento; la evaluación final se realiza en el conjunto de prueba.

In [4]:
TERMINALS = [
    'monto_alto', 'hora_inusual', 'distancia_lejana',
    'ingreso_bajo', 'monto_muy_alto',
]
POPULATION_SIZE = 250
GENERATIONS = 50
MAX_TREE_HEIGHT = 7

def build_features(frame):
    amount_ratio = frame['monto_pesos'] / frame['ingreso_mensual_pesos']
    return np.column_stack([
        amount_ratio >= 0.20,
        (frame['hora_dia'] <= 5) | (frame['hora_dia'] >= 23),
        frame['distancia_residencia_km'] >= 20,
        frame['ingreso_mensual_pesos'] <= 1_800_000,
        amount_ratio >= 0.50,
    ]).astype(bool)

def boolean_not(value):
    return not value

def build_primitive_set():
    primitive_set = gp.PrimitiveSet('FRAUD_RULE', len(TERMINALS))
    for index, name in enumerate(TERMINALS):
        primitive_set.renameArguments(**{f'ARG{index}': name})
    primitive_set.addPrimitive(operator.and_, 2, name='AND')
    primitive_set.addPrimitive(operator.or_, 2, name='OR')
    primitive_set.addPrimitive(operator.xor, 2, name='XOR')
    primitive_set.addPrimitive(boolean_not, 1, name='NOT')
    return primitive_set

random.seed(SEED)
np.random.seed(SEED)
train_x = build_features(train)
test_x = build_features(test)
train_y = train[TARGET].to_numpy(dtype=int)
test_y = test[TARGET].to_numpy(dtype=int)
primitive_set = build_primitive_set()

if not hasattr(creator, 'FraudFitness'):
    creator.create('FraudFitness', base.Fitness, weights=(-1.0, -0.01))
if not hasattr(creator, 'FraudIndividual'):
    creator.create('FraudIndividual', gp.PrimitiveTree, fitness=creator.FraudFitness)

toolbox = base.Toolbox()
toolbox.register('expression', gp.genHalfAndHalf, pset=primitive_set, min_=1, max_=3)
toolbox.register('individual', tools.initIterate, creator.FraudIndividual, toolbox.expression)
toolbox.register('population', tools.initRepeat, list, toolbox.individual)

def evaluate(individual):
    rule = gp.compile(individual, primitive_set)
    predictions = np.array([bool(rule(*row)) for row in train_x], dtype=int)
    false_negatives = np.sum((train_y == 1) & (predictions == 0))
    false_positives = np.sum((train_y == 0) & (predictions == 1))
    weighted_errors = 2 * false_negatives + false_positives
    return float(weighted_errors), float(len(individual))

toolbox.register('evaluate', evaluate)
toolbox.register('select', tools.selTournament, tournsize=4)
toolbox.register('mate', gp.cxOnePoint)
toolbox.register('expr_mut', gp.genFull, min_=0, max_=2)
toolbox.register('mutate', gp.mutUniform, expr=toolbox.expr_mut, pset=primitive_set)
toolbox.decorate('mate', gp.staticLimit(
    key=operator.attrgetter('height'), max_value=MAX_TREE_HEIGHT
))
toolbox.decorate('mutate', gp.staticLimit(
    key=operator.attrgetter('height'), max_value=MAX_TREE_HEIGHT
))

population = toolbox.population(n=POPULATION_SIZE)
hall_of_fame = tools.HallOfFame(1)
algorithms.eaSimple(
    population, toolbox, cxpb=0.7, mutpb=0.25, ngen=GENERATIONS,
    halloffame=hall_of_fame, verbose=False,
)
best = hall_of_fame[0]
rule = gp.compile(best, primitive_set)
predictions = np.array([bool(rule(*row)) for row in test_x], dtype=int)
print(f'Regla encontrada: {best}')
print(f'Tamaño de la regla: {len(best)} nodos')
print(f'ROC-AUC: {roc_auc_score(test_y, predictions):.3f}')
print(f'F1 de fraude: {f1_score(test_y, predictions, zero_division=0):.3f}')
print('Matriz de confusión [real x predicho]:')
print(confusion_matrix(test_y, predictions))
print(classification_report(
    test_y, predictions, target_names=['No fraude', 'Fraude'], zero_division=0,
))

Regla encontrada: AND(hora_inusual, monto_muy_alto)
Tamaño de la regla: 3 nodos
ROC-AUC: 0.500
F1 de fraude: 0.000
Matriz de confusión [real x predicho]:
[[1228    0]
 [  22    0]]
              precision    recall  f1-score   support

   No fraude       0.98      1.00      0.99      1228
      Fraude       0.00      0.00      0.00        22

    accuracy                        0.98      1250
   macro avg       0.49      0.50      0.50      1250
weighted avg       0.97      0.98      0.97      1250


## 4. Interpretación

Compara ROC-AUC, F1 y las matrices de confusión de ambos enfoques. La regresión logística usa variables numéricas estandarizadas; la regla evolutiva usa indicadores discretos y prioriza reducir falsos negativos. Ninguna métrica por sí sola basta para tomar decisiones operativas, y la regla aprendida refleja las simplificaciones del generador sintético.